In [2]:
import os
import cv2
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

from skimage.feature import hog

In [3]:
data_dir = "data"   # 🔁 เปลี่ยนเป็น path dataset คุณ
img_size = 128      # เพิ่มจาก 64 → ได้ detail มากขึ้น

X = []
y = []

classes = sorted(os.listdir(data_dir))
print("Classes:", classes)

for label in classes:
    folder = os.path.join(data_dir, label)

    for img_name in tqdm(os.listdir(folder), desc=label):
        img_path = os.path.join(folder, img_name)

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.resize(img, (img_size, img_size))
        X.append(img)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

Classes: ['Auto Rickshaws', 'Bikes', 'Cars', 'Motorcycles', 'Planes', 'Ships', 'Trains']


Trains: 100%|██████████| 800/800 [00:02<00:00, 289.66it/s]

Dataset shape: (5590, 128, 128, 3)


In [4]:
le = LabelEncoder()
y = le.fit_transform(y)

print("Encoded classes:", le.classes_)

Encoded classes: ['Auto Rickshaws' 'Bikes' 'Cars' 'Motorcycles' 'Planes' 'Ships' 'Trains']


In [5]:
X_hog = []

for img in tqdm(X, desc="Extracting HOG"):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    feature = hog(
        gray,
        orientations=12,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

    X_hog.append(feature)

X_features = np.array(X_hog)

print("HOG shape:", X_features.shape)

Extracting HOG: 100%|██████████| 5590/5590 [00:25<00:00, 217.29it/s]


HOG shape: (5590, 10800)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (4472, 10800)
Test size: (1118, 10800)


In [7]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling complete")

Scaling complete


In [8]:
pca = PCA(n_components=0.95)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print("After PCA:", X_train.shape)

After PCA: (4472, 1979)


In [9]:
print("Training SVM...")

svm = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    probability=True
)

svm.fit(X_train, y_train)

svm_pred = svm.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

Training SVM...
SVM Accuracy: 0.8094812164579607


In [10]:
print("Training RandomForest...")

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
print("RandomForest Accuracy:", accuracy_score(y_test, rf_pred))

Training RandomForest...
RandomForest Accuracy: 0.7164579606440071


In [11]:
print("Training XGBoost...")

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss'
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))

Training XGBoost...
XGBoost Accuracy: 0.7835420393559929


In [12]:
print("Training Voting Classifier...")

voting = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('svm', svm),
        ('xgb', xgb)
    ],
    voting='soft'
)

voting.fit(X_train, y_train)

vote_pred = voting.predict(X_test)

print("Voting Accuracy:", accuracy_score(y_test, vote_pred))
print(classification_report(y_test, vote_pred, target_names=le.classes_))

Training Voting Classifier...
Voting Accuracy: 0.8228980322003577
                precision    recall  f1-score   support

Auto Rickshaws       0.83      0.78      0.80       160
         Bikes       0.99      0.93      0.95       160
          Cars       0.86      0.73      0.79       158
   Motorcycles       0.87      0.81      0.84       160
        Planes       0.86      0.81      0.84       160
         Ships       0.64      0.93      0.76       160
        Trains       0.82      0.78      0.80       160

      accuracy                           0.82      1118
     macro avg       0.84      0.82      0.83      1118
  weighted avg       0.84      0.82      0.83      1118



In [13]:
import joblib

joblib.dump(voting, "vehicle_voting_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(pca, "pca.pkl")
joblib.dump(le, "label_encoder.pkl")

print("Models saved ✅")

Models saved ✅
